# RT-DETR COCO Colab 평가
런타임에서 GPU를 선택하면 빠릅니다. 먼저 로컬에서 `export_coco_artifact.py`로 만든 `.pth` 파일 하나를 준비하세요. FP32 기준 모델, 고정 COCO 데이터, 공식 RT-DETR 소스는 아래 셀이 내려받습니다. 같은 세션에서 기준과 양자화 모델을 차례로 평가합니다.

**주의:** 현재 W4A4/W6A6 결과는 fake quantization입니다. 코랩 속도·메모리·파일 크기를 실제 정수 배포 성능으로 해석하지 마세요.


In [ ]:
import os, subprocess, sys, pathlib
repo = pathlib.Path('/content/haiteam4_finetune')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/foxyaef/haiteam4_finetune.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
import torch, torchvision
print('PyTorch', torch.__version__, 'torchvision', torchvision.__version__, 'GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
subprocess.run([sys.executable, 'scripts/colab_setup.py'], check=True)
subprocess.run([sys.executable, 'prepare_data.py'], check=True)
subprocess.run([sys.executable, 'scripts/coco_run.py', 'check'], check=True)


## 양자화 모델 파일 하나 넣기
아래 셀은 로컬 `.pth` 파일을 업로드합니다. 큰 파일이면 Google Drive를 마운트한 뒤 `artifact_path`를 Drive 파일 경로로 바꿔도 됩니다. 업로드 파일은 신뢰할 수 있는 팀원이 이 프로젝트의 exporter로 만든 파일만 사용하세요.


In [ ]:
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('정확히 하나의 .pth 파일을 업로드하세요')
name = next(iter(uploaded))
if not name.endswith('.pth'):
    raise ValueError('export_coco_artifact.py로 만든 .pth 파일이 필요합니다')
artifact_path = str(pathlib.Path(name).resolve())
del uploaded
print('Artifact:', artifact_path)


In [ ]:
import datetime
tag = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S')
baseline_id, artifact_id = 'fp32_' + tag, 'artifact_' + tag
for model_name, run_id in [('official', baseline_id), (artifact_path, artifact_id)]:
    subprocess.run([sys.executable, 'scripts/colab_eval.py', '--model', model_name, '--run-id', run_id, '--device', 'auto'], check=True)
print('Completed:', baseline_id, artifact_id)


In [ ]:
import zipfile
bundle = repo / ('colab_results_' + tag + '.zip')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as zipped:
    for run_id in (baseline_id, artifact_id):
        for name in ('metrics.json', 'class_metrics.csv', 'profile.json', 'provenance.json'):
            file = repo / 'runs' / 'colab_eval' / run_id / name
            zipped.write(file, arcname=run_id + '/' + name)
files.download(str(bundle))
print('로컬에서 ZIP의 두 결과 폴더를 분석하세요:', bundle.name)
